## Orchestration
Agent orchestration is the process of coordinating agents, tools, handoffs, and execution flow to accomplish a task efficiently and autonomously.

First agentic ai Project with openai agents SDK framework:
<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h3 style="color:#fff800;">Project Idea - Sending Emails</h2>
            <span style="color:#fff800;">We are going to write an agent that sends emails. The best way to do this is with an email provider
            like SendGrid or Resend. But this requires a fair amount of heavy-lifting: you need to send from a proper email host where you own the domain and you can prove it with DNS records. It's quite a hassle.<br/><br/>
            So we're going to do it the free & simple way: using SMTP server configuration to send direct from your email. It's an easy setup, but it's not as powerful (eg you can't receive emails). If you'd like to take this further, use SendGrid or Resend instead..<br/><br/>
            And actually the email send is optional. We're doing this to demonstrate an Agent sending emails. The important part is the Agent work, not the email sending. Feel free to swap out the function for a Pushover push if you'd prefer.
            </span>
        </td>
    </tr>
</table>

### Part1: Email Setup

### Part2: Orchestration via Code

### Part3: Orchestration via LLM's
- 3.a: LLM's with tools
- 3.b: LLM's with handoff


In [34]:
from dotenv import load_dotenv
import requests
from agents import Agent,trace,Runner, function_tool, SQLiteSession, OpenAIChatCompletionsModel ,set_tracing_disabled
import os
import asyncio
import smtplib
from email.message import EmailMessage
from openai.types.responses import ResponseTextDeltaEvent
from agents.extensions.visualization import draw_graph
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent

In [36]:
load_dotenv(override=True)


True

In [37]:
#gemini api key setup with openai client library (or openai ask)
gemini_api_key=os.getenv("GEMINI_API_KEY")
gemini_base_url=os.getenv("GEMINI_BASE_URL")

gemini_client= AsyncOpenAI(api_key=gemini_api_key, base_url=gemini_base_url)

gemini_model= OpenAIChatCompletionsModel(
    model="gemini-3.5-flash-lite",
    openai_client=gemini_client
)


In [38]:
set_tracing_disabled(True)

In [ ]:
EMAIL_ADDRESS= os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER=os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD= os.getenv("EMAIL_APP_PASSWORD")

USE_EMAIL= EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Mail is setup , now we are ready to use it.")
else:
    print("Mail is not setup, We'll send push notification instead.")


In [ ]:
def send_mails(subject, text_body, html_body):
    msg = EmailMessage()
    msg["from"]=EMAIL_ADDRESS
    msg["to"]=EMAIL_ADDRESS
    msg["subject"]=subject
    msg.set_content(text_body)
    msg.add_alternative(html_body,subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER,587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS,EMAIL_APP_PASSWORD)
        server.send_message(msg)


In [ ]:
send_mails("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")


#### Our fallback strategy-> send push notification to mobile

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    payload ={ "user":pushover_user , "token":pushover_token, "message":message }
    requests.post(pushover_url, data=payload)


In [ ]:
def send_message(mesasge, text_body, html_body):
    if USE_EMAIL:
        send_mails(mesasge,text_body,html_body)
    else:
        push(f"Subject: {mesasge}\n\n Body:{text_body}")

In [ ]:
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

### Agent Orchestration
There are two way for agent orchestration
- One is via code 
- Other one is via LLM's 

By code : More predictable, deterministic   

By LLM's : More autonomous & Powerfull




### Orchestrating by CODE

In [28]:
intro =""" You are sales agent working for Karan.ai comppany 
         a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
         You write  mails """


instruction1= intro + "Your email style is professional, serious, with gravitas and credibility."
instruction2= intro + "Your email style is witty, engaging and humorous"
instruction3= intro + "Your email style is concise, to the point in style of busy senior executive"



In [32]:
sales_agent1= Agent(name="Sales_agent1", instructions=instruction1, model=gemini_model)
sales_agent2= Agent(name="Sales_agent2", instructions=instruction2, model=gemini_model)
sales_agent3= Agent(name="Sales_agent3", instructions=instruction3, model=gemini_model)



In [39]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email.")
async for event in result.stream_events():
    if event.type=="raw_response_event" and isinstance(event.data , ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True) 
        
        


Subject: Navigating SOC 2 readiness without the resource drain

Dear [Prospect Name],

For most engineering and security leaders, SOC 2 compliance is viewed as a necessary tax—months of policy drafting, evidence gathering, and continuous monitoring that pull valuable resources away from core product development. 

Traditional consulting firms offer a high-touch, high-cost solution, while legacy compliance tools still demand significant manual intervention to map controls and remediate gaps. 

At Karan.ai, we have fundamentally re-engineered this process. 

Our AI-powered platform automates up to 80% of the compliance lifecycle—from intelligent gap analysis and automated evidence collection to real-time risk remediation. We provide the rigor and assurance required by top-tier auditors, but with the velocity and precision that modern engineering teams demand.

We help organizations achieve readiness in weeks, not months, while drastically reducing audit preparation costs.

If reducing fr

In [ ]:
message= "write a cold sales mail."

with trace("Mails are ready"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

# outputs=[]
# for result in results:
#     outputs.append(result.final_output)

outputs= [ result.final_output for result in results] # this is list comprehension 



In [41]:
for output in outputs:
    print(output + "\n\n")

Subject: The compliance bottleneck in your scaling strategy

Dear [Prospect Name],

As [Prospect Company] continues to scale its operations, the friction between accelerating feature deployment and meeting stringent security frameworks inevitably intensifies. 

For most engineering and compliance leaders, SOC 2 preparation is a zero-sum game: it either diverts critical developer hours away from your core product roadmap, or it introduces costly delays to enterprise sales cycles. Traditional consulting firms offer slow, manual remediation, while standard checklist tools still leave the heavy lifting of evidence gathering and policy drafting to your internal team.

Karan.ai was built to fundamentally eliminate this friction. 

Our AI-powered SaaS platform integrates directly into your infrastructure to automate up to 80% of the SOC 2 compliance lifecycle. By continuously collecting evidence, preemptively flagging security drift, and autonomously drafting audit-ready documentation, Karan.

In [42]:
decision = """ 
          You pick the best sale cold email from the given option.
          Imagine you are a customer and pick the one you are most likely to respond to.
          Do not give an explanation; reply with the selected email only.
"""


In [43]:
sales_mail_picker= Agent( name="Sales_mail_picker", instructions=decision, model=gemini_model)


In [44]:
message = "write cold sales email."

with trace("Mails are ready to pickup:"):
    results= await asyncio.gather(
        Runner.run(sales_agent1,message),
        Runner.run(sales_agent2,message),
        Runner.run(sales_agent3,message)

    )

outputs= [result.final_output for result in results]

emails = "cold Sales email are  : \n\n "+ " \n\nEmail:\n\n ".join(outputs)

best_email= await Runner.run(sales_mail_picker,emails)

print(best_email.final_output)




Subject: SOC2 compliance in weeks, not months?

[First Name],

Most engineering leaders I speak with dread SOC2 prep. It's usually 3-4 months of manual evidence collection, endless spreadsheets, and expensive consultants. 

At Karan.ai, we built an AI-powered SaaS platform that automates 80% of this. We connect to your stack, continuously collect evidence, and prep you for the auditor in a fraction of the time.

Worth a 10-minute look this week to see how much time you can save? 

Best,

[Your Name]  
Sales, Karan.ai


### now add tools to send the mails autonoumasly
here we only need to add @function_tool decorator, it will create junky json of that method and create the tool handler by itself

In [45]:
@function_tool
def send_email_tool(subject:str, text_body:str, html_body:str)->str:
     """
     Send out an email with the given subject and body to all sales prospects 

     Args:
        subject: The subject of the email
        text_body:The body of the email as plain text
        html_body: The HTML body of the email

     """
     send_message(subject,text_body,html_body)
     return " email send successfully"


In [46]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}